In [1]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download('nudratabbas/global-ads-performance-google-meta-tiktok')
print(path)

100%|██████████| 58.8k/58.8k [00:00<00:00, 43.1MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/nudratabbas/global-ads-performance-google-meta-tiktok/versions/1


In [2]:
import pandas as pd
import os

path = '/root/.cache/kagglehub/datasets/nudratabbas/global-ads-performance-google-meta-tiktok/versions/1'
print(os.listdir(path))

['global_ads_performance_dataset.csv']


In [6]:
df = pd.read_csv(path + '/global_ads_performance_dataset.csv')
df.info()
df.head()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1800 non-null   object 
 1   platform       1800 non-null   object 
 2   campaign_type  1800 non-null   object 
 3   industry       1800 non-null   object 
 4   country        1800 non-null   object 
 5   impressions    1800 non-null   int64  
 6   clicks         1800 non-null   int64  
 7   CTR            1800 non-null   float64
 8   CPC            1800 non-null   float64
 9   ad_spend       1800 non-null   float64
 10  conversions    1800 non-null   int64  
 11  CPA            1800 non-null   float64
 12  revenue        1800 non-null   float64
 13  ROAS           1800 non-null   float64
dtypes: float64(6), int64(3), object(5)
memory usage: 197.0+ KB


,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
count,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000
mean,102919.018889,3962.675556,0.038427,1.572756,6171.527272,181.562222,46.608961,30101.850450,6.450367
std,55740.900690,2941.858037,0.017082,0.800872,5776.996958,171.424239,41.185556,34560.032941,6.590986
min,5059.000000,91.000000,0.008900,0.280000,58.000000,2.000000,4.800000,142.690000,0.130000
25%,54948.000000,1678.000000,0.025400,0.950000,1966.587500,59.000000,20.202500,7275.757500,2.170000
50%,103653.000000,3318.000000,0.035550,1.460000,4393.860000,130.000000,33.375000,18362.965000,4.295000
75%,150470.250000,5628.000000,0.049800,2.050000,8455.830000,252.250000,56.812500,38963.385000,8.212500
max,199650.000000,16660.000000,0.095600,3.950000,38453.320000,1151.000000,335.860000,295028.260000,49.000000


In [7]:
import pandas as pd
import numpy as np

# Convert date to datetime for time-based analysis
df['date'] = pd.to_datetime(df['date'])

# --- Step 1: Add CPM (not in original dataset) ---
# CPM = cost per 1000 impressions
df['CPM'] = (df['ad_spend'] / df['impressions']) * 1000

# --- Step 2: Verify given metrics instead of blindly trusting them ---
# This shows analytical rigor — recompute CTR and CPC ourselves and check they match
df['CTR_check'] = df['clicks'] / df['impressions']
df['CPC_check'] = df['ad_spend'] / df['clicks']

ctr_mismatch = (abs(df['CTR'] - df['CTR_check']) > 0.001).sum()
cpc_mismatch = (abs(df['CPC'] - df['CPC_check']) > 0.01).sum()
print(f"CTR mismatches: {ctr_mismatch} / {len(df)}")
print(f"CPC mismatches: {cpc_mismatch} / {len(df)}")

# Drop the check columns once verified
df.drop(columns=['CTR_check', 'CPC_check'], inplace=True)

# --- Step 3: Platform-level performance summary ---
platform_summary = df.groupby('platform').agg(
    total_spend=('ad_spend', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_CTR=('CTR', 'mean'),
    avg_CPC=('CPC', 'mean'),
    avg_CPM=('CPM', 'mean'),
    avg_CPA=('CPA', 'mean'),
    avg_ROAS=('ROAS', 'mean'),
    total_conversions=('conversions', 'sum')
).sort_values('avg_ROAS', ascending=False)
print("\n=== Platform Performance ===")
print(platform_summary)

# --- Step 4: Campaign type performance ---
campaign_summary = df.groupby('campaign_type').agg(
    avg_CTR=('CTR', 'mean'),
    avg_ROAS=('ROAS', 'mean'),
    total_spend=('ad_spend', 'sum')
).sort_values('avg_ROAS', ascending=False)
print("\n=== Campaign Type Performance ===")
print(campaign_summary)

# --- Step 5: Industry performance ---
industry_summary = df.groupby('industry').agg(
    avg_CTR=('CTR', 'mean'),
    avg_ROAS=('ROAS', 'mean'),
    avg_CPA=('CPA', 'mean')
).sort_values('avg_ROAS', ascending=False)
print("\n=== Industry Performance ===")
print(industry_summary)

# --- Step 6: Top and bottom 5 performing campaigns by ROAS ---
top5 = df.nlargest(5, 'ROAS')[['platform', 'campaign_type', 'industry', 'country', 'ROAS', 'ad_spend']]
bottom5 = df.nsmallest(5, 'ROAS')[['platform', 'campaign_type', 'industry', 'country', 'ROAS', 'ad_spend']]
print("\n=== Top 5 by ROAS ===")
print(top5)
print("\n=== Bottom 5 by ROAS (underperformers to flag) ===")
print(bottom5)

# --- Step 7: Correlation check ---
# Does higher spend correlate with better ROAS, or does it plateau?
corr = df[['ad_spend', 'CTR', 'CPC', 'CPM', 'ROAS', 'CPA']].corr()
print("\n=== Correlation Matrix ===")
print(corr)

# Save cleaned dataset for use in the Streamlit app
df.to_csv('ad_campaign_cleaned.csv', index=False)
print("\nCleaned file saved: ad_campaign_cleaned.csv")

CTR mismatches: 0 / 1800
CPC mismatches: 0 / 1800

=== Platform Performance ===
            total_spend  total_revenue   avg_CTR   avg_CPC    avg_CPM  \
platform                                                                
TikTok Ads   2653418.51    20223540.07  0.054963  1.009978  55.306844   
Meta Ads     2106061.67    11926045.79  0.024983  1.316095  32.911172   
Google Ads   6349268.91    22033744.95  0.039856  2.149069  85.960146   

              avg_CPA  avg_ROAS  total_conversions  
platform                                            
TikTok Ads  29.196711  9.538600             122452  
Meta Ads    39.096921  6.915730              73262  
Google Ads  64.064653  4.113028             131098  

=== Campaign Type Performance ===
                avg_CTR  avg_ROAS  total_spend
campaign_type                                 
Search         0.039426  7.000273   2868006.85
Display        0.038091  6.448667   2644735.12
Video          0.037630  6.340000   2796458.48
Shopping       0.03

In [8]:
from google.colab import files
files.download('ad_campaign_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>